<a href="https://colab.research.google.com/github/ShrutiPatel263/AeroCare/blob/main/FD001_AllFeatures_WeightedResults_Done.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

=== Cross-Validation Summary ===

MAE: 9.9078,

 RMSE: 13.3048,

  R²: 0.8945

Best Fold: 3

✅ Weighted Ensemble Test Results:

  MAE:  10.7311

  RMSE: 14.6268
  
  R²:   0.8726

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Pre-processing

In [ ]:
import numpy as np
import pandas as pd

from IPython.display import display, HTML
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio


import seaborn as sns
from importlib import reload
import matplotlib.pyplot as plt
import matplotlib
import warnings

# Configure Jupyter Notebook
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 500)
pd.set_option('display.expand_frame_repr', False)
# pd.set_option('max_colwidth', -1)
display(HTML("<style>div.output_scroll { height: 35em; }</style>"))

reload(plt)
%matplotlib inline
%config InlineBackend.figure_format ='retina'

warnings.filterwarnings('ignore')

# configure plotly graph objects
pio.renderers.default = 'iframe'
# pio.renderers.default = 'vscode'

pio.templates["ck_template"] = go.layout.Template(
    layout_colorway = px.colors.sequential.Viridis,
#     layout_hovermode = 'closest',
#     layout_hoverdistance = -1,
    layout_autosize=False,
    layout_width=800,
    layout_height=600,
    layout_font = dict(family="Calibri Light"),
    layout_title_font = dict(family="Calibri"),
    layout_hoverlabel_font = dict(family="Calibri Light"),
#     plot_bgcolor="white",
)

# pio.templates.default = 'seaborn+ck_template+gridon'
pio.templates.default = 'ck_template+gridon'
# pio.templates.default = 'seaborn+gridon'
# pio.templates

In [ ]:
index_names = ['engine', 'cycle']
setting_names = ['setting_1', 'setting_2', 'setting_3']
sensor_names=[ "(Fan inlet temperature) (◦R)",
"(LPC outlet temperature) (◦R)",
"(HPC outlet temperature) (◦R)",
"(LPT outlet temperature) (◦R)",
"(Fan inlet Pressure) (psia)",
"(bypass-duct pressure) (psia)",
"(HPC outlet pressure) (psia)",
"(Physical fan speed) (rpm)",
"(Physical core speed) (rpm)",
"(Engine pressure ratio(P50/P2)",
"(HPC outlet Static pressure) (psia)",
"(Ratio of fuel flow to Ps30) (pps/psia)",
"(Corrected fan speed) (rpm)",
"(Corrected core speed) (rpm)",
"(Bypass Ratio) ",
"(Burner fuel-air ratio)",
"(Bleed Enthalpy)",
"(Required fan speed)",
"(Required fan conversion speed)",
"(High-pressure turbines Cool air flow)",
"(Low-pressure turbines Cool air flow)" ]
col_names = index_names + setting_names + sensor_names

In [ ]:
df_train = pd.read_csv('drive/MyDrive/CMaps/train_FD001.txt',sep=r'\s+',header=None,index_col=False,names=col_names)
df_test = pd.read_csv('drive/MyDrive/CMaps/test_FD001.txt',sep=r'\s+',header=None,index_col=False,names=col_names)
df_test_RUL = pd.read_csv('drive/MyDrive/CMaps/RUL_FD001.txt',sep=r'\s+',header=None,index_col=False,names=['RUL'])

In [ ]:
keep_features = [
    "(Fan inlet temperature) (◦R)",
    "(LPC outlet temperature) (◦R)",
    "(HPC outlet temperature) (◦R)",
    "(LPT outlet temperature) (◦R)",
    "(Fan inlet Pressure) (psia)",
    "(bypass-duct pressure) (psia)",
    "(HPC outlet pressure) (psia)",
    "(Physical fan speed) (rpm)",
    "(Physical core speed) (rpm)",
    "(Engine pressure ratio(P50/P2)",
    "(HPC outlet Static pressure) (psia)",
    "(Ratio of fuel flow to Ps30) (pps/psia)",
    "(Corrected fan speed) (rpm)",
    "(Corrected core speed) (rpm)",
    "(Bypass Ratio) ",
    "(Burner fuel-air ratio)",
    "(Bleed Enthalpy)",
    "(Required fan speed)",
    "(Required fan conversion speed)",
    "(High-pressure turbines Cool air flow)",
    "(Low-pressure turbines Cool air flow)",
    "setting_1",
    "setting_2",
    "setting_3"
]

In [ ]:
features = list(keep_features)

In [ ]:
# define the maximum life of each engine, as this could be used to obtain the RUL at each point in time of the engine's life
df_train_RUL = df_train.groupby(['engine']).agg({'cycle':'max'})
df_train_RUL.rename(columns={'cycle':'life'},inplace=True)
df_train_RUL.head()

,life
engine,
1,192
2,287
3,179
4,189
5,269


In [ ]:
df_train=df_train.merge(df_train_RUL,how='left',on=['engine'])

In [ ]:
df_train['RUL']=df_train['life']-df_train['cycle']
df_train.drop(['life'],axis=1,inplace=True)

# the RUL prediction is only useful nearer to the end of the engine's life, therefore we put an upper limit on the RUL
# this is a bit sneaky, since it supposes that the test set has RULs of less than this value, the closer you are
# to the true value, the more accurate the model will be
df_train['RUL'][df_train['RUL']>125]=125
df_train.head()

,engine,cycle,setting_1,setting_2,setting_3,(Fan inlet temperature) (◦R),(LPC outlet temperature) (◦R),(HPC outlet temperature) (◦R),(LPT outlet temperature) (◦R),(Fan inlet Pressure) (psia),(bypass-duct pressure) (psia),(HPC outlet pressure) (psia),(Physical fan speed) (rpm),(Physical core speed) (rpm),(Engine pressure ratio(P50/P2),(HPC outlet Static pressure) (psia),(Ratio of fuel flow to Ps30) (pps/psia),(Corrected fan speed) (rpm),(Corrected core speed) (rpm),(Bypass Ratio),(Burner fuel-air ratio),(Bleed Enthalpy),(Required fan speed),(Required fan conversion speed),(High-pressure turbines Cool air flow),(Low-pressure turbines Cool air flow),RUL
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,21.61,554.36,2388.06,9046.19,1.3,47.47,521.66,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190,125
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,21.61,553.75,2388.04,9044.07,1.3,47.49,522.28,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236,125
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,21.61,554.26,2388.08,9052.94,1.3,47.27,522.42,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442,125
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,21.61,554.45,2388.11,9049.48,1.3,47.13,522.86,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739,125
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,21.61,554.00,2388.06,9055.15,1.3,47.28,522.19,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044,125


In [ ]:
def create_sequences(df, window_size, stride, engine_col='engine'):
    sequences, targets, engine_ids = [], [], []

    for engine_id in df[engine_col].unique():
        engine_data = df[df[engine_col] == engine_id].sort_values('cycle')
        feature_cols = [col for col in engine_data.columns if col not in ['engine', 'cycle', 'RUL']]

        values = engine_data[feature_cols].values
        rul_values = engine_data['RUL'].values

        for i in range(0, len(values) - window_size + 1, stride):
            sequences.append(values[i:i + window_size])
            targets.append(rul_values[i + window_size - 1])
            engine_ids.append(engine_id)

    return np.array(sequences), np.array(targets), np.array(engine_ids)



# Scale features
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
feature_cols = [col for col in df_train.columns if col not in ['engine', 'cycle', 'RUL']]
df_train[feature_cols] = scaler.fit_transform(df_train[feature_cols])
df_test[feature_cols] = scaler.transform(df_test[feature_cols])

In [ ]:
pip install tensorflow

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, regularizers
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [ ]:
np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
# ============================================================
# ADVANCED RUL PREDICTION: Gated Ensemble BiLSTM + Attention (Optimized)
# ============================================================

import tensorflow as tf
from tensorflow.keras import layers, models, regularizers, callbacks, optimizers
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# MODIFIED create_sequences function to handle missing 'RUL' column for test set
def create_sequences(df, window_size, stride, engine_col='engine'):
    sequences, targets, engine_ids = [], [], []

    # Check if 'RUL' column exists in the DataFrame
    has_rul = 'RUL' in df.columns

    for engine_id in df[engine_col].unique():
        engine_data = df[df[engine_col] == engine_id].sort_values('cycle')
        feature_cols = [col for col in engine_data.columns if col not in ['engine', 'cycle', 'RUL']]

        values = engine_data[feature_cols].values

        if has_rul:
            rul_values = engine_data['RUL'].values
        else:
            # If RUL column is not present (e.g., for test data), create a dummy array.
            # These 'targets' will be discarded for X_test anyway.
            rul_values = np.array([])

        # Ensure there are enough data points for the window
        if len(values) < window_size:
            continue # Skip engines that are too short for a single sequence

        for i in range(0, len(values) - window_size + 1, stride):
            sequences.append(values[i:i + window_size])
            if has_rul: # Only append RUL target if RUL column exists
                targets.append(rul_values[i + window_size - 1])
            engine_ids.append(engine_id)

    return np.array(sequences), np.array(targets), np.array(engine_ids)


CONFIG = {
    'WINDOW_SIZE': 60,        # FD001 engines avg ~200 cycles; 30 is sharper
    'STRIDE': 1,
    'LSTM_UNITS': [128, 64],
    'DENSE_UNITS': 80,
    'DROPOUT_RATE': 0.2,      # Reduced — your CV→test gap suggests mild overfit
    'L2_REG': 0.0005,         # Slightly relaxed
    'BATCH_SIZE': 32,         # Smaller batch → better generalization
    'EPOCHS': 200,
    'LEARNING_RATE': 0.001,
    'PATIENCE': 20,           # Tighter early stopping
    'N_SPLITS': 5
}

print("\nOptimized Configuration:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")


X_train, y_train, train_engine_ids = create_sequences(
    df_train, CONFIG['WINDOW_SIZE'], CONFIG['STRIDE']
)
X_test, _, test_engine_ids = create_sequences(
    df_test, CONFIG['WINDOW_SIZE'], CONFIG['STRIDE']
)

print(f"Training sequences: {X_train.shape}")
print(f"Test sequences: {X_test.shape}")


# ============================================================
# Custom Layers
# ============================================================

class GatingLayer(layers.Layer):
    def __init__(self, num_branches, **kwargs):
        super(GatingLayer, self).__init__(**kwargs)
        self.num_branches = num_branches

    def build(self, input_shape):
        self.gate_weights = self.add_weight(
            shape=(input_shape[0][-1], self.num_branches),
            initializer='glorot_uniform',
            trainable=True
        )
        self.gate_bias = self.add_weight(
            shape=(self.num_branches,),
            initializer='zeros',
            trainable=True
        )

    def call(self, inputs):
        input_stats = tf.reduce_mean(inputs[0], axis=1)
        gate_scores = tf.nn.softmax(tf.matmul(input_stats, self.gate_weights) + self.gate_bias, axis=-1)
        gate_scores = tf.expand_dims(tf.expand_dims(gate_scores, 1), -1)
        stacked = tf.stack(inputs, axis=2)
        gated = stacked * gate_scores
        fused = tf.reduce_sum(gated, axis=2)
        return fused


class AttentionLayer(layers.Layer):
    """Attention mechanism to highlight critical timesteps after fusion."""
    def __init__(self, **kwargs):
        super(AttentionLayer, self).__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(shape=(input_shape[-1], input_shape[-1]),
                                 initializer="glorot_uniform", trainable=True)
        self.b = self.add_weight(shape=(input_shape[-1],),
                                 initializer="zeros", trainable=True)
        self.u = self.add_weight(shape=(input_shape[-1],),
                                 initializer="glorot_uniform", trainable=True)

    def call(self, x):
        u_t = tf.tanh(tf.tensordot(x, self.W, axes=1) + self.b)
        attn_scores = tf.nn.softmax(tf.tensordot(u_t, self.u, axes=1), axis=1)
        attn_scores = tf.expand_dims(attn_scores, -1)
        context = tf.reduce_sum(x * attn_scores, axis=1)
        return context


# ============================================================
# Optimized Gated BiLSTM + Attention Model
# ============================================================

def build_gated_attention_bilstm(input_shape, config):
    inputs = layers.Input(shape=input_shape, name='input')

    # Parallel branches: LSTM, GRU, CNN
    branch_lstm = layers.LSTM(64, return_sequences=True,
                              kernel_regularizer=regularizers.l2(config['L2_REG']))(inputs)
    branch_lstm = layers.LayerNormalization()(branch_lstm)

    branch_gru = layers.GRU(64, return_sequences=True,
                            kernel_regularizer=regularizers.l2(config['L2_REG']))(inputs)
    branch_gru = layers.LayerNormalization()(branch_gru)

    branch_cnn = layers.Conv1D(64, 5, padding='same', activation='relu',
                               kernel_regularizer=regularizers.l2(config['L2_REG']))(inputs)
    branch_cnn = layers.Conv1D(64, 3, padding='same', activation='relu',
                               kernel_regularizer=regularizers.l2(config['L2_REG']))(branch_cnn)
    branch_cnn = layers.LayerNormalization()(branch_cnn)

    fused = GatingLayer(num_branches=3)([branch_lstm, branch_gru, branch_cnn])
    fused = layers.Dropout(config['DROPOUT_RATE'])(fused)

    # BiLSTM layers for sequential modeling
    x = layers.Bidirectional(
        layers.LSTM(config['LSTM_UNITS'][0],
                    return_sequences=True,
                    kernel_regularizer=regularizers.l2(config['L2_REG'])),
        name='bilstm_1'
    )(fused)
    x = layers.LayerNormalization()(x)
    x = layers.Dropout(config['DROPOUT_RATE'])(x)

    x = layers.Bidirectional(
        layers.LSTM(config['LSTM_UNITS'][1],
                    return_sequences=True,
                    kernel_regularizer=regularizers.l2(config['L2_REG'])),
        name='bilstm_2'
    )(x)
    x = layers.LayerNormalization()(x)
    x = layers.Dropout(config['DROPOUT_RATE'])(x)

    # Attention layer
    context = AttentionLayer()(x)

    # Dense head
    x = layers.Dense(config['DENSE_UNITS'], activation='relu')(context)
    x = layers.Dropout(config['DROPOUT_RATE'])(x)

    outputs = layers.Dense(1, activation='linear', name='rul_output')(x)

    model = models.Model(inputs, outputs, name='Gated_Attention_BiLSTM')

    opt = optimizers.Adam(learning_rate=config['LEARNING_RATE'])
    model.compile(
        optimizer=opt,
        loss=tf.keras.losses.Huber(delta=12.0),
        metrics=['mae', 'mse']
    )
    return model


# ============================================================
# Train with Advanced Callbacks
# ============================================================

# ============================================================
# Train with Advanced Callbacks
# ============================================================

def train_model_with_cv(X_train, y_train, engine_ids, input_shape, config, n_splits=5):
    from sklearn.model_selection import GroupKFold

    gkf = GroupKFold(n_splits=n_splits)
    results, models_list = [], []

    for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_train, y_train, engine_ids), 1):
        print(f"\n{'='*25} FOLD {fold}/{n_splits} {'='*25}")
        X_tr, X_val = X_train[tr_idx], X_train[val_idx]
        y_tr, y_val = y_train[tr_idx], y_train[val_idx]

        model = build_gated_attention_bilstm(input_shape, config)

        cb = [
            callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.4, patience=6, min_lr=5e-6),
            callbacks.EarlyStopping(monitor='val_loss', patience=config['PATIENCE'],
                                    restore_best_weights=True),
            callbacks.ModelCheckpoint(f"best_fold_{fold}.h5", save_best_only=True)
        ]

        history = model.fit(
            X_tr, y_tr,
            validation_data=(X_val, y_val),
            epochs=config['EPOCHS'],
            batch_size=config['BATCH_SIZE'],
            verbose=1,
            callbacks=cb
        )

        val_loss, val_mae, val_mse = model.evaluate(X_val, y_val, verbose=0)
        val_rmse = np.sqrt(val_mse)

        y_pred_val = model.predict(X_val, verbose=0).flatten()
        val_r2 = r2_score(y_val, y_pred_val)

        results.append({'fold': fold, 'mae': val_mae, 'rmse': val_rmse, 'r2': val_r2})
        models_list.append(model)

        print(f"Fold {fold} Results -> MAE: {val_mae:.4f}, RMSE: {val_rmse:.4f}, R²: {val_r2:.4f}")

    print("\n=== Cross-Validation Summary ===")
    avg_mae  = np.mean([r['mae']  for r in results])
    avg_rmse = np.mean([r['rmse'] for r in results])
    avg_r2   = np.mean([r['r2']   for r in results])
    print(f"MAE: {avg_mae:.4f}, RMSE: {avg_rmse:.4f}, R²: {avg_r2:.4f}")

    best_model_idx = np.argmax([r['r2'] for r in results])
    print(f"\nBest Fold: {best_model_idx + 1}")

    return models_list, results   # ← returns ALL models now


# ============================================================
# Execute Training
# ============================================================

input_shape = (X_train.shape[1], X_train.shape[2])
all_models, cv_results = train_model_with_cv(
    X_train, y_train, train_engine_ids, input_shape, CONFIG, CONFIG['N_SPLITS']
)

# ============================================================
# Ensemble ALL folds for test prediction
# ============================================================

# ============================================================
# Weighted Ensemble by fold R² score
# ============================================================

fold_r2_scores = np.array([r['r2'] for r in cv_results])
fold_weights = fold_r2_scores / fold_r2_scores.sum()  # normalize to sum=1

print("\nFold weights:")
for i, (r2, w) in enumerate(zip(fold_r2_scores, fold_weights), 1):
    print(f"  Fold {i}: R²={r2:.4f}, weight={w:.4f}")

all_preds = []
for m in all_models:
    preds = m.predict(X_test, verbose=0).flatten()
    preds = np.clip(preds, 0, None)
    all_preds.append(preds)

# Weighted average
y_test_pred_all = np.average(all_preds, axis=0, weights=fold_weights)

unique_engines = sorted(set(test_engine_ids))
y_true, y_pred = [], []
for eng_id in unique_engines:
    mask = test_engine_ids == eng_id
    y_pred.append(y_test_pred_all[mask][-1])
    y_true.append(df_test_RUL.iloc[int(eng_id) - 1]['RUL'])

y_true, y_pred = np.array(y_true), np.array(y_pred)
test_mae  = mean_absolute_error(y_true, y_pred)
test_rmse = np.sqrt(mean_squared_error(y_true, y_pred))
test_r2   = r2_score(y_true, y_pred)

print(f"\n✅ Weighted Ensemble Test Results:")
print(f"  MAE:  {test_mae:.4f}")
print(f"  RMSE: {test_rmse:.4f}")
print(f"  R²:   {test_r2:.4f}")

if test_r2 >= 0.88:
    print("\n🏆 EXCELLENT PERFORMANCE ACHIEVED!")



Optimized Configuration:
  WINDOW_SIZE: 60
  STRIDE: 1
  LSTM_UNITS: [128, 64]
  DENSE_UNITS: 80
  DROPOUT_RATE: 0.2
  L2_REG: 0.0005
  BATCH_SIZE: 32
  EPOCHS: 200
  LEARNING_RATE: 0.001
  PATIENCE: 20
  N_SPLITS: 5
Training sequences: (14731, 60, 24)
Test sequences: (7351, 60, 24)

========================= FOLD 1/5 =========================
Epoch 1/200
369/369 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - loss: 294.5132 - mae: 29.6918 - mse: 1812.3927

369/369 ━━━━━━━━━━━━━━━━━━━━ 37s 39ms/step - loss: 294.1395 - mae: 29.6596 - mse: 1809.4819 - val_loss: 86.8262 - val_mae: 12.1867 - val_mse: 223.1312 - learning_rate: 0.0010
Epoch 2/200
369/369 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - loss: 59.8931 - mae: 9.0814 - mse: 156.4298

369/369 ━━━━━━━━━━━━━━━━━━━━ 13s 34ms/step - loss: 59.8847 - mae: 9.0806 - mse: 156.4006 - val_loss: 71.8341 - val_mae: 10.4788 - val_mse: 185.8433 - learning_rate: 0.0010
Epoch 3/200
368/369 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - loss: 46.3616 - mae: 7.7300 - mse: 112.8492

369/369 ━━━━━━━━━━━━━━━━━━━━ 13s 35ms/step - loss: 46.3546 - mae: 7.7292 - mse: 112.8280 - val_loss: 69.6046 - val_mae: 10.3767 - val_mse: 168.2966 - learning_rate: 0.0010
Epoch 4/200
369/369 ━━━━━━━━━━━━━━━━━━━━ 12s 33ms/step - loss: 37.4609 - mae: 6.7267 - mse: 86.7429 - val_loss: 77.7675 - val_mae: 11.1582 - val_mse: 196.6268 - learning_rate: 0.0010
Epoch 5/200
369/369 ━━━━━━━━━━━━━━━━━━━━ 21s 34ms/step - loss: 33.8654 - mae: 6.3753 - mse: 76.4472 - val_loss: 83.6979 - val_mae: 11.5154 - val_mse: 217.1730 - learning_rate: 0.0010
Epoch 6/200
369/369 ━━━━━━━━━━━━━━━━━━━━ 13s 35ms/step - loss: 32.7587 - mae: 6.1719 - mse: 73.5698 - val_loss: 79.6739 - val_mae: 11.0659 - val_mse: 205.7100 - learning_rate: 0.0010
Epoch 7/200
369/369 ━━━━━━━━━━━━━━━━━━━━ 13s 34ms/step - loss: 29.9061 - mae: 5.8668 - mse: 65.2434 - val_loss: 82.8197 - val_mae: 11.0168 - val_mse: 221.5796 - learning_rate: 0.0010
Epoch 8/200
369/369 ━━━━━━━━━━━━━━━━━━━━ 13s 35ms/step - loss: 27.9883 - mae: 5.6305 - mse: 60.9

369/369 ━━━━━━━━━━━━━━━━━━━━ 23s 38ms/step - loss: 293.5998 - mae: 29.5993 - mse: 1802.8191 - val_loss: 89.3538 - val_mae: 12.0514 - val_mse: 260.5735 - learning_rate: 0.0010
Epoch 2/200
368/369 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - loss: 70.1410 - mae: 10.1194 - mse: 188.2132

...

369/369 ━━━━━━━━━━━━━━━━━━━━ 22s 36ms/step - loss: 329.0374 - mae: 32.6355 - mse: 2090.4707 - val_loss: 81.6718 - val_mae: 10.9489 - val_mse: 239.6705 - learning_rate: 0.0010
Epoch 2/200
369/369 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - loss: 72.8151 - mae: 10.3627 - mse: 202.9154

369/369 ━━━━━━━━━━━━━━━━━━━━ 12s 32ms/step - loss: 72.7943 - mae: 10.3607 - mse: 202.8345 - val_loss: 70.6366 - val_mae: 10.1638 - val_mse: 199.1574 - learning_rate: 0.0010
Epoch 3/200
369/369 ━━━━━━━━━━━━━━━━━━━━ 12s 34ms/step - loss: 50.8253 - mae: 8.2309 - mse: 125.7379 - val_loss: 73.4354 - val_mae: 10.3999 - val_mse: 213.8666 - learning_rate: 0.0010
Epoch 4/200
369/369 ━━━━━━━━━━━━━━━━━━━━ 12s 33ms/step - loss: 42.3108 - mae: 7.2850 - mse: 99.5162 - val_loss: 72.8495 - val_mae: 10.3937 - val_mse: 211.1155 - learning_rate: 0.0010
Epoch 5/200
369/369 ━━━━━━━━━━━━━━━━━━━━ 12s 34ms/step - loss: 34.7342 - mae: 6.4810 - mse: 78.5357 - val_loss: 80.2328 - val_mae: 11.1929 - val_mse: 228.5033 - learning_rate: 0.0010
Epoch 6/200
369/369 ━━━━━━━━━━━━━━━━━━━━ 12s 34ms/step - loss: 32.9517 - mae: 6.3019 - mse: 72.6562 - val_loss: 83.4885 - val_mae: 11.3114 - val_mse: 257.4086 - learning_rate: 0.0010
Epoch 7/200
369/369 ━━━━━━━━━━━━━━━━━━━━ 12s 34ms/step - loss: 29.8737 - mae: 5.9221 - mse: 64

369/369 ━━━━━━━━━━━━━━━━━━━━ 24s 39ms/step - loss: 314.4337 - mae: 31.4256 - mse: 1917.1799 - val_loss: 93.2810 - val_mae: 12.5211 - val_mse: 266.3472 - learning_rate: 0.0010
Epoch 2/200
369/369 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - loss: 86.8281 - mae: 11.6920 - mse: 249.4091

369/369 ━━━━━━━━━━━━━━━━━━━━ 14s 37ms/step - loss: 86.8113 - mae: 11.6904 - mse: 249.3505 - val_loss: 77.6704 - val_mae: 11.0688 - val_mse: 203.7243 - learning_rate: 0.0010
Epoch 3/200
369/369 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - loss: 59.8174 - mae: 9.0897 - mse: 154.1852

369/369 ━━━━━━━━━━━━━━━━━━━━ 19s 35ms/step - loss: 59.8090 - mae: 9.0889 - mse: 154.1572 - val_loss: 71.8253 - val_mae: 10.4071 - val_mse: 194.7818 - learning_rate: 0.0010
Epoch 4/200
369/369 ━━━━━━━━━━━━━━━━━━━━ 13s 34ms/step - loss: 45.9776 - mae: 7.6808 - mse: 111.0229 - val_loss: 74.4238 - val_mae: 10.5545 - val_mse: 199.6373 - learning_rate: 0.0010
Epoch 5/200
369/369 ━━━━━━━━━━━━━━━━━━━━ 13s 34ms/step - loss: 38.7105 - mae: 6.9658 - mse: 88.3451 - val_loss: 85.7303 - val_mae: 11.6188 - val_mse: 243.9374 - learning_rate: 0.0010
Epoch 6/200
369/369 ━━━━━━━━━━━━━━━━━━━━ 13s 34ms/step - loss: 35.0108 - mae: 6.5356 - mse: 77.7989 - val_loss: 86.1004 - val_mae: 11.4902 - val_mse: 259.1507 - learning_rate: 0.0010
Epoch 7/200
369/369 ━━━━━━━━━━━━━━━━━━━━ 13s 34ms/step - loss: 32.3148 - mae: 6.2186 - mse: 71.2933 - val_loss: 87.4930 - val_mae: 11.5628 - val_mse: 261.0401 - learning_rate: 0.0010
Epoch 8/200
369/369 ━━━━━━━━━━━━━━━━━━━━ 13s 34ms/step - loss: 28.8000 - mae: 5.8380 - mse: 61.